# [선택] 예제 08. 금융 차트 자동 생성 에이전트
### 자연어로 차트를 지정하여 분석 결과 시각화 · 2일 차 · 시간이 남을 때 25~40분

> 📌 **선택 예제입니다.** 07까지 진도가 맞고 시간이 남을 때 진행합니다. 08을 건너뛰어도 09 · 10은 그대로 진행되며, 차트는 09 보고서와 10 통합 실행에서 자동으로 만들어집니다.

> **실행 방법**: 위에서부터 순서대로 실행하세요. 셀을 클릭하고 `Shift + Enter`. AI 답변이 나오는 셀은 몇 초~수십 초 걸립니다. 셀 왼쪽이 `[*]`이면 아직 실행 중입니다.

| 항목 | 내용 |
|---|---|
| 데이터 | 예제 07과 같은 `data/bank_customers.csv` |
| 오늘의 결과물 | 고객관리 · 예금 유치 분석 차트 3종 (PNG 이미지 + 마우스로 보는 HTML) + AI 차트 설명 |

**오늘 해결할 질문**: "말로 요청하면 보고서에 넣을 차트와 설명이 바로 나올까?"

## 0. 실습 준비

FinanceAgent를 불러오고, 이 PC의 로컬 AI(Ollama)가 켜져 있는지 확인합니다.
`AI 연결 확인: qwen3:4b-instruct 준비 완료`가 나오면 성공입니다.

In [ ]:
import json, urllib.request
from pathlib import Path
from finance_agent import FinanceAgent
import llm_config

DATA = "data/bank_customers.csv"
models = [m["name"] for m in json.load(urllib.request.urlopen("http://127.0.0.1:11434/api/tags"))["models"]]
print("데이터 파일:", Path(DATA).is_file())
print("AI 연결 확인:", llm_config.OLLAMA_MODEL, "준비 완료" if llm_config.OLLAMA_MODEL in models else "→ 모델이 없습니다. 강사에게 알려 주세요")
agent = FinanceAgent(provider="ollama")

## 08-1. 요청 문장을 어떻게 알아듣나?

에이전트는 요청을 쉼표로 나누고, 정해진 낱말 목록으로 **기준 · 지표 · 형태**를 고릅니다. AI에게 맡기지 않아 같은 요청이면 늘 같은 차트가 나옵니다.

| 고르는 것 | 요청 속 낱말 → 결과 |
|---|---|
| 기준 | 연령·나이 → 연령대 / 직업 → 직업군 / 잔액 → 잔액구간 / 대출 → 대출상태 / 캠페인 → 과거캠페인 |
| 지표 | 고객 수·분포·인원 → 고객수 / 그 외 → 가입률(%) |
| 형태 | 가로 → 가로막대 / 원그래프·파이·비중 → 원(고객수만) / 그 외 → 막대 |

In [ ]:
from exercises import ex08_chart

요청 = "연령대별 가입률 막대그래프, 잔액 구간별 고객 분포, 직업군별 가입률 가로막대"
ex08_chart.parse_request(요청)

## 08-2. 차트 3종 만들기 ⭐

In [ ]:
결과 = agent.run("ex08", DATA, 요청)

> 💬 AI 답변의 문장은 PC와 모델에 따라 조금씩 다릅니다. 표·숫자·파일은 Python이 계산하므로 모두 같게 나옵니다.

In [ ]:
from IPython.display import Image, display

for f in 결과["artifacts"]:
    if f.endswith(".png"):
        display(Image(filename=f, width=560))

> 💡 PNG는 보고서·Excel에 넣는 그림, HTML은 브라우저로 열면 **마우스를 올려 값을 보는** 차트입니다(1일 차 Plotly와 같음). `outputs` 폴더에서 HTML 파일을 더블클릭해 보세요.

## 08-3. 🔧 요청 바꿔 보기 (실습)

아래 요청을 고쳐 다른 차트를 만들어 봅니다. 예: `"대출 상태별 가입률 가로막대, 과거 캠페인별 고객 분포 원그래프"`

In [ ]:
요청2 = "대출 상태별 가입률 가로막대, 과거 캠페인별 고객 분포 원그래프"     # ← 바꿔 보세요
print(ex08_chart.parse_request(요청2))
결과 = agent.run("ex08", DATA, 요청2)
for f in 결과["artifacts"]:
    if f.endswith(".png"):
        display(Image(filename=f, width=560))

## 08-4. AI 차트 설명 검토하기 ⚠️

- 설명 속 숫자가 차트의 막대 값과 같은지 확인합니다 (숫자 검증 줄 참고).
- "가장 높다/낮다"가 맞는지 그림과 대조합니다.
- 고객 수가 적은 막대(예: 과거캠페인 "이전 가입" 1,511명)에 대한 단정적 표현이 없는지 봅니다.

## 08-5. [선택] 만든 차트를 Excel에 붙이기

In [ ]:
from openpyxl import load_workbook
from openpyxl.drawing.image import Image as XLImage

wb = load_workbook("outputs/고객별_예금가입_분석.xlsx")          # 예제 07에서 만든 파일
sh = wb.create_sheet("차트")
for i, f in enumerate([f for f in 결과["artifacts"] if f.endswith(".png")]):
    img = XLImage(f)
    img.width, img.height = 560, 320
    sh.add_image(img, f"A{1 + i * 18}")
wb.save("outputs/고객별_예금가입_분석_차트포함.xlsx")
print("저장: outputs/고객별_예금가입_분석_차트포함.xlsx")

## ✅ 오늘 배운 것

| 하고 싶은 일 | 코드 |
|---|---|
| 요청이 어떻게 해석되는지 보기 | `ex08_chart.parse_request("요청")` |
| 요청 → 차트 + AI 설명 | `agent.run("ex08", DATA, "요청")` |
| 노트북에 그림 띄우기 | `display(Image(filename="파일.png"))` |

## 📝 복습 미션

> 아래 셀에서 **`# ←` 표시가 있는 줄만** 바꿔 실행하세요. 막히면 맨 아래 **정답 셀**을 실행해 결과를 확인합니다.

**미션.** `"나이별 고객 수 원그래프, 잔액별 가입률"`이 어떻게 해석되는지 먼저 `parse_request`로 확인한 뒤 실행해 보세요.
**이렇게 나오면 성공!** `[{'기준': '연령대', '지표': '고객수', '형태': '원'}, {'기준': '잔액구간', '지표': '가입률(%)', '형태': '막대'}]`

**다음 예제 전에 이것만은 꼭!**
- [ ] 차트 종류는 정해진 목록으로 해석 → 같은 요청이면 같은 차트
- [ ] AI 설명은 그림과 대조해서 검토

In [ ]:
# 미션
요청3 = "연령대별 가입률 막대그래프"     # ← "나이별 고객 수 원그래프, 잔액별 가입률"로
print(ex08_chart.parse_request(요청3))
결과 = agent.run("ex08", DATA, 요청3)

### 정답 (먼저 위 셀을 바꿔 실행해 본 뒤 확인하세요)

In [ ]:
ex08_chart.parse_request("나이별 고객 수 원그래프, 잔액별 가입률")

---
### ▶ 다음 예제 09. 금융 업무보고서 · 공문 작성 에이전트
지금까지의 표와 차트, AI 문장을 모아 **Word 업무보고서와 내부 기안문**을 만듭니다.